# Magnetic Resonance Imaging II
 
## Exercise 4
 
---
 
**Prof. Dr. Markus Zimmermann**
 
**Mohamed Wassim Bensalem**
 
---

In Exercises 2 and 3 you worked with Cartesian k-space trajectories and saw how acceleration interacts with image quality. In Exercise 2, $R=2$ Cartesian undersampling was handled cleanly by parallel imaging. In Exercise 3, $R=2$ was not just a convenience — it was necessary to shorten EPI's long readout train and suppress off-resonance artifacts. But pushing Cartesian acceleration further to $R=4$ broke the reconstruction in both cases: the aliasing becomes too severe for parallel imaging alone to handle.
 
Further acceleration therefore requires moving away from regular Cartesian undersampling. This exercise introduces **radial sampling**: instead of parallel lines, each TR acquires a single spoke through the center of k-space at a different angle. At $R=4$, Cartesian undersampling produces coherent aliasing ghosts that are difficult to remove. Radial undersampling at the same rate produces incoherent, noise-like streaking artifacts instead — and incoherence is exactly what **Compressed Sensing (CS)** reconstruction exploits.
 
CS adds a Total Variation (TV) regularization term to the reconstruction objective, promoting images that are sparse in the gradient domain — piecewise smooth, with sharp edges. You will implement CS via the **Alternating Direction Method of Multipliers (ADMM)**, which solves the regularized problem by alternating between a CG data-consistency step and a proximal soft-thresholding step.
 
Because radial samples no longer lie on a Cartesian grid, a standard FFT cannot be used — reconstruction requires the **Non-Uniform Fourier Transform (NUFFT)**. You will start with $R=1$ to verify the sequence and reconstruction pipeline, then increase to $R=4$ to see where CG alone fails and CS succeeds.
 
New modules in this exercise:
 
- **`mri.conopt.functional.fourier`** — provides `nonuniform_fourier_transform_forward` and `nonuniform_fourier_transform_adjoint`, the NUFFT primitives.
- **`mri.conopt.linops.composite`** — provides `ConcatOperator`, which stacks multiple operators vertically into a single operator.

## Software Imports
 
All required libraries and modules are imported here.

In [ ]:
%matplotlib widget

import sys
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch
from tqdm import tqdm

sys.path.insert(0, "/home/jovyan/mri/src")
sys.path.insert(0, "/home/jovyan/shared-folder/Magnetresonanztomographie-2/mri/src")

from mri.conopt.functional.fourier import (
    nonuniform_fourier_transform_adjoint,
    nonuniform_fourier_transform_forward,
)
from mri.conopt.linops import LinearOperator
from mri.conopt.linops.composite import ConcatOperator
from mri.parim.analytic import cyclic_head_coil

## System and Sequence Parameters
 
`R` controls the undersampling factor: the sequence acquires `n_y // R` radial spokes instead of the full `n_y`. Start with `R = 1` to verify the full pipeline, then increase to `R = 4` to observe the streaking artifacts and see how CS suppresses them. `num_dummy_pulses` sets the number of preparatory TRs played before data acquisition begins — see the Sequence Components section for details.

In [ ]:
seq_filename = "gre_radial.seq"

fov_x, fov_y = 256e-3, 256e-3
n_x, n_y = 128, 128
n_channels = 8
R = 1
num_dummy_pulses = 10

case = ["T1-weighted", "T2-weighted", "PD-weighted"][0]

if case == "T1-weighted":
    flip_angle_deg = 90
    tr = 250e-3
    te = 5e-3
elif case == "T2-weighted":
    flip_angle_deg = 40
    tr = 5000e-3
    te = 50e-3
elif case == "PD-weighted":
    flip_angle_deg = 40
    tr = 5000e-3
    te = 5e-3

torch.set_default_device("cpu")

## System and Sequence Object

In [ ]:
system = pp.Opts(
    max_grad=28,
    grad_unit="mT/m",
    max_slew=150,
    slew_unit="T/m/s",
)

seq = pp.Sequence(system)

## Linear Operators

The `ChannelOperator` is identical to Exercises 2 and 3 and is provided here.

In [ ]:
class ChannelOperator(LinearOperator):

    def __init__(
        self,
        coil_sensitivities: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nC = coil_sensitivities.shape[0]

        self.coil_sensitivities = coil_sensitivities
        self.device = device
        self.dtype = torch.float32
        self.internal_dtype = torch.complex64
        self.forward_shape = (1, *data_shape)
        self.adjoint_shape = (nC, *data_shape)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.forward_shape)
        y = self.coil_sensitivities * x
        return y.view(torch.float32).ravel()

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.adjoint_shape)
        y = torch.sum(torch.conj(self.coil_sensitivities) * x, dim=0, keepdim=True)
        return y.view(torch.float32).ravel()

## Numerical Solvers

The CG solver is identical to Exercises 2 and 3 and is provided here. A `disable_progressbar` flag has been added — this is used inside ADMM to suppress the inner CG progress bar.

In [ ]:
def conjugate_gradient(
    Q: LinearOperator,
    b: torch.Tensor,
    x: torch.Tensor,
    maxiter: int = 10,
    disable_progressbar: bool = False,
) -> torch.Tensor:

    g_prev = torch.ones(x.shape, dtype=x.dtype)
    p_prev = torch.zeros(x.shape, dtype=x.dtype)

    for iteration in tqdm(
        range(maxiter),
        disable=disable_progressbar,
    ):

        g = Q @ x - b

        conjugacy_factor = g @ g / (g_prev @ g_prev)
        p = g + conjugacy_factor * p_prev
        stepsize = g @ g / (p @ (Q @ p))
        x = x - stepsize * p

        g_prev = g.clone()
        p_prev = p.clone()

    return x

## Sequence Components
 
The radial GRE sequence uses the same RF pulse, ADC, and timing logic as the Cartesian GRE in Exercise 1. The key difference is that there are no fixed `gx` and `gy` gradients — instead, a single readout gradient `g_ro` and pre-phaser `g_pre` are defined along the x-axis, and their areas are projected onto x and y for each spoke angle in the sequence loop. The spoiler is applied in both x and y to ensure complete dephasing regardless of the spoke angle.
 
These components are provided here so you can focus on the sequence assembly and reconstruction.

In [ ]:
# Your solution here


delta_kx = 1 / fov_x
delta_ky = 1 / fov_y
raster_eps = 1e-12

rf = pp.make_sinc_pulse(
    flip_angle=np.deg2rad(flip_angle_deg),
    system=system,
    use="excitation",
)

g_ro = pp.make_trapezoid(
    channel="x",
    flat_area=n_x * delta_kx,
    flat_time=3.2e-3,
    system=system,
)

g_pre = pp.make_trapezoid(
    channel="x",
    area=-g_ro.area / 2,
    system=system,
)

adc = pp.make_adc(
    num_samples=n_x,
    duration=g_ro.flat_time,
    delay=g_ro.rise_time,
    system=system,
)

gx_spoiler = pp.make_trapezoid(
    channel="x",
    area=4 * g_ro.area,
    system=system,
)

gy_spoiler = pp.make_trapezoid(
    channel="y",
    area=4 * g_ro.area,
    system=system,
)

te_delay = (
    te
    - (pp.calc_duration(rf) - pp.calc_rf_center(rf)[0])
    - pp.calc_duration(g_pre)
    - pp.calc_duration(g_ro) / 2
)

tr_delay = (
    tr
    - pp.calc_duration(rf)
    - pp.calc_duration(g_pre)
    - pp.calc_duration(g_ro)
    - pp.calc_duration(gx_spoiler)
    - te_delay
)

te_delay = (
    np.ceil((te_delay - raster_eps) / seq.grad_raster_time) * seq.grad_raster_time
)
tr_delay = (
    np.ceil((tr_delay - raster_eps) / seq.grad_raster_time) * seq.grad_raster_time
)

assert np.all(te_delay >= 0)
assert np.all(tr_delay >= 0)

## Task 1: (Undersampled) Radial Sequence Assembly

In a radial sequence, each TR acquires one spoke through the center of k-space. The spoke angle for spoke $i$ is:
 
$$\theta_i = \frac{i \cdot \pi}{n_y / R}, \qquad i = 0, 1, \ldots, \frac{n_y}{R} - 1$$
 
In code, this is simply:
 
```python
angles = np.linspace(0, np.pi, num=n_y // R, endpoint=False)
```
 
covering angles from $0$ to $\pi$ (a full half-rotation, since opposite angles cover the same spoke). For each angle, the pre-phaser and readout gradients are rotated by projecting `g_pre.area` and `g_ro.area` onto the x and y axes:
 
$$g_{x,\text{pre}} = \cos(\theta) \cdot g_\text{pre}, \qquad g_{y,\text{pre}} = \sin(\theta) \cdot g_\text{pre}$$
$$g_{x,\text{ro}} = \cos(\theta) \cdot g_\text{ro}, \qquad g_{y,\text{ro}} = \sin(\theta) \cdot g_\text{ro}$$
 
**Dummy pulses.** Before acquiring data, the sequence plays `num_dummy_pulses` preparatory TRs to drive the magnetization into a steady state. These TRs are identical to the acquisition TRs — same RF pulse, same gradients — but with the ADC switched off so no data is recorded. Without them, the first few spokes would be acquired with a different magnetization state than the rest, introducing inconsistencies in k-space.
 
The cleanest way to implement this is to extend the loop range to start at `-num_dummy_pulses` instead of `0`. For negative indices the ADC is omitted; for non-negative indices the ADC is included. The spoke angle is fixed at $\theta = 0$ for all dummy pulses (any angle works — what matters is that the RF and gradients are played):
 
```python
for index in range(-num_dummy_pulses, n_y // R):
 
    if index < 0:
        angle = 0
    else:
        angle = angles[index]
 
    ...
 
    if index < 0:
        seq.add_block(gx_ro, gy_ro)        # dummy: no ADC
    else:
        seq.add_block(gx_ro, gy_ro, adc)   # acquisition: ADC on
```
 
**Your task:** Assemble the radial GRE sequence using the structure above. First compute `angles` using `np.linspace` over $[0, \pi)$ with `n_y // R` points. Then build the loop: for each iteration, construct the angle-dependent `gx_pre`, `gy_pre`, `gx_ro`, `gy_ro` using `pp.make_trapezoid` with the projected areas and durations from `g_pre` and `g_ro`. Assemble each TR with RF, pre-phasers, TE delay, readout (with or without ADC), spoilers, and TR delay.

In [ ]:
# Your solution here


angles = np.linspace(0, np.pi, num=n_y // R, endpoint=False)

for index in range(-num_dummy_pulses, n_y // R):

    if index < 0:
        angle = 0
    else:
        angle = angles[index]

    seq.add_block(rf)

    gx_pre = pp.make_trapezoid(
        channel="x",
        area=np.cos(angle) * g_pre.area,
        duration=pp.calc_duration(g_pre),
        system=system,
    )

    gy_pre = pp.make_trapezoid(
        channel="y",
        area=np.sin(angle) * g_pre.area,
        duration=pp.calc_duration(g_pre),
        system=system,
    )

    gx_ro = pp.make_trapezoid(
        channel="x",
        area=np.cos(angle) * g_ro.area,
        duration=pp.calc_duration(g_ro),
        system=system,
    )

    gy_ro = pp.make_trapezoid(
        channel="y",
        area=np.sin(angle) * g_ro.area,
        duration=pp.calc_duration(g_ro),
        system=system,
    )

    seq.add_block(gx_pre, gy_pre)

    seq.add_block(pp.make_delay(te_delay))

    if index < 0:
        seq.add_block(gx_ro, gy_ro)  # dummy: no ADC
    else:
        seq.add_block(gx_ro, gy_ro, adc)  # acquisition: ADC on

    seq.add_block(gx_spoiler, gy_spoiler)

    seq.add_block(pp.make_delay(tr_delay))

## Verification and Export

In [ ]:
ok, error_report = seq.check_timing()

if ok:
    print("Timing check passed successfully")
else:
    print("Timing check failed. Error listing follows:")
    [print(e) for e in error_report]

plt.figure()
seq.plot(time_range=(0.0, tr - tr_delay + te_delay))

seq.write(seq_filename, v141_compat=True)

## Phantom and Multi-Channel Simulation

Run this cell as-is.

In [ ]:
phantom = mr0.util.load_phantom(size=(n_x, n_y))

coil_sensitivities_recon = cyclic_head_coil(
    (n_channels, n_x, n_y, 1),
).to(torch.complex64)
coil_sensitivities_recon = coil_sensitivities_recon.flip((1, 2))
coil_sensitivities_recon = coil_sensitivities_recon[..., 0]

coil_sensitivities_sim = cyclic_head_coil(
    (n_channels, n_x, n_y, 1),
    fov_scaling=(phantom.size[0] / fov_x, phantom.size[1] / fov_y, 1),
).to(torch.complex64)
phantom.coil_sens = coil_sensitivities_sim

plt.figure()
phantom.plot()

mr0_seq = mr0.Sequence.import_file(seq_filename)

data = phantom.build()
graph = mr0.compute_graph(mr0_seq, data)
signal = mr0.execute_graph(graph, mr0_seq, data, print_progress=True)

## Task 2: Coil Images

Reshape `signal` and permute to $(n_\text{channels}, n_K)$ as in the previous exercises. Call `.contiguous()` after permuting — the NUFFT library requires the data to be stored in contiguous memory.
 
Instead of an inverse FFT, use `nonuniform_fourier_transform_adjoint(k_traj_adc, data, (n_x, n_y))` to reconstruct each coil image from the non-Cartesian samples. It returns a tensor of shape $(n_\text{channels},\ n_x,\ n_y,\ 1)$ — drop the trailing $z$-dimension with `[..., 0]`.
 
The k-space trajectory `k_traj_adc` is extracted from the sequence with `seq.calculate_kspace()`. The first element `kspace[0]` has shape $(3, n_K)$, with rows corresponding to $k_x$, $k_y$, $k_z$. Keep only the first two rows with `[:2, :]`, normalize to $[-0.5, 0.5]$ by dividing by the maximum value and multiplying by $0.5$, and convert to `torch.float32`.
 
**Your tasks:**
 
1. Reshape and permute `signal` to obtain `data` of shape $(n_\text{channels}, n_K)$, then call `.contiguous()`.
2. Extract `k_traj_adc`, keep only $k_x$ and $k_y$, normalize to $[-0.5, 0.5]$, and convert to `torch.float32`.
3. Call `nonuniform_fourier_transform_adjoint` and drop the trailing dimension to get `coil_images`.
4. Display the coil images and coil sensitivity maps in 2×4 grids.

In [ ]:
# Your solution here


data = signal.clone()
data = data.reshape((-1, n_channels)).permute((1, 0))

data = data.contiguous()

kspace = seq.calculate_kspace()
k_traj_adc = kspace[0]
k_traj_adc = k_traj_adc[:2, :]
k_traj_adc = k_traj_adc / k_traj_adc.max() * 0.5
k_traj_adc = torch.tensor(k_traj_adc, dtype=torch.float32)

coil_images = nonuniform_fourier_transform_adjoint(k_traj_adc, data, (n_x, n_y))[..., 0]

plt.figure()
for i in range(n_channels):
    plt.subplot(2, 4, i + 1)
    plt.imshow(torch.abs(coil_images[i, :, :]).numpy().T, cmap="gray")
    plt.title(f"Channel {i + 1}")
plt.suptitle("Coil images")
plt.tight_layout()
plt.show()

plt.figure()
for i in range(n_channels):
    plt.subplot(2, 4, i + 1)
    plt.imshow(torch.abs(coil_sensitivities_recon[i, :, :]).numpy().T, cmap="viridis")
    plt.title(f"Channel {i + 1}")
plt.suptitle("Coil sensitivity maps")
plt.tight_layout()
plt.show()

## Task 3: The Nonuniform Fourier Transform Operator

You will now implement `NonuniformFourierTransformOperator`, which wraps the NUFFT primitives into the `LinearOperator` framework so that it can be composed with `ChannelOperator` and used inside CG and ADMM.
 
**Shapes.** The operator maps a multi-channel image to multi-channel non-Cartesian k-space samples:
 
$$\mathbf{F}_\text{nu}:\quad (n_\text{channels},\ n_x,\ n_y,\ 1) \;\rightarrow\; (n_\text{channels},\ n_K)$$
 
where $n_K = n_x \cdot (n_y / R)$ is the total number of acquired k-space samples (spokes × samples per spoke). Note the trailing size-1 dimension in the input shape: `nonuniform_fourier_transform_forward` and `nonuniform_fourier_transform_adjoint` expect a 3D spatial volume with shape $(n_x,\ n_y,\ n_z)$, and for 2D imaging you pass $n_z = 1$. You must therefore store `forward_shape = (n_channels, n_x, n_y, 1)` and add this trailing dimension accordingly. The adjoint shape is `(n_channels, n_K)`.
 
Since the forward and adjoint shapes differ, the operator is not square:
 
```python
self.shape = (
    2 * n_channels * n_K,   # output size of forward (k-space, real repr.)
    2 * n_channels * n_x * n_y * 1,  # input size of forward (image, real repr.)
)
```
 
**Forward (`_matvec`):** reshape the input to `forward_shape`, call `nonuniform_fourier_transform_forward(self.k, x)`, return the output raveled as `float32`.
 
**Adjoint (`_rmatvec`):** reshape the input to `adjoint_shape`, call `nonuniform_fourier_transform_adjoint(self.k, x, self.n_modes)`, where `self.n_modes = (n_x, n_y)` — the 2D image size. The function returns a tensor of shape $(n_\text{channels},\ n_x,\ n_y,\ 1)$; ravel and return as `float32`.
 
**Your task:** Implement `NonuniformFourierTransformOperator`. Store `k`, `forward_shape`, `adjoint_shape`, `n_modes`, `shape`, `dtype`, and `internal_dtype` in the constructor.

In [ ]:
# Your solution here


class NonuniformFourierTransformOperator(LinearOperator):
    def __init__(
        self,
        k: torch.Tensor,
        input_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):

        nD, nK = k.shape

        nC, nX, nY = input_shape
        self.n_modes = (nX, nY)

        self.k = k
        # Important: add nZ dimension
        self.forward_shape = (*input_shape, 1)
        self.adjoint_shape = (nC, nK)

        self.shape = (
            2 * torch.prod(torch.tensor(self.adjoint_shape)),
            2 * torch.prod(torch.tensor(self.forward_shape)),
        )

        # Set the data type to float32 and internal dtype to complex64
        self.dtype = torch.float32
        self.internal_dtype = torch.complex64
        self.device = device

    def _matvec(
        self,
        input_tensor: torch.Tensor,
    ) -> torch.Tensor:
        input_tensor = input_tensor.view(self.internal_dtype).reshape(
            self.forward_shape
        )
        output_tensor = nonuniform_fourier_transform_forward(
            self.k,
            input_tensor,
        )
        return output_tensor.ravel().view(self.dtype)

    def _rmatvec(
        self,
        input_tensor: torch.Tensor,
    ) -> torch.Tensor:
        input_tensor = input_tensor.view(self.internal_dtype).reshape(
            self.adjoint_shape
        )
        output_tensor = nonuniform_fourier_transform_adjoint(
            self.k,
            input_tensor,
            self.n_modes,
        )
        return output_tensor.ravel().view(self.dtype)

## Task 4: Compose the Forward Operator and Compute Regridded Image

The forward operator for radial parallel imaging is $\mathbf{A} = \mathbf{F}_\text{nu} \mathbf{C}$, where $\mathbf{C}$ maps the single-channel image to multi-channel, and $\mathbf{F}_\text{nu}$ applies the NUFFT to each channel. Note that there is no mask operator $\mathbf{M}$ here — the trajectory $\mathbf{k}$ already encodes which k-space locations are sampled.
 
**Your tasks:**
 
1. Instantiate `F` as a `NonuniformFourierTransformOperator` with `k_traj_adc` and `input_shape = (n_channels, n_x, n_y)`.
2. Instantiate `C` as a `ChannelOperator` with `coil_sensitivities_recon` and `data_shape = (n_x, n_y)`.
3. Compose `A = F @ C`.
4. Compute `AHd = A.H @ data.view(torch.float32).ravel()`.
5. Reshape and display the magnitude of `AHd`.

In [ ]:
# Your solution here

F = NonuniformFourierTransformOperator(k_traj_adc, (n_channels, n_x, n_y))
C = ChannelOperator(coil_sensitivities_recon, (n_x, n_y))

A = F @ C

AHd = A.H @ data.view(torch.float32).ravel()

AHd_image = AHd.view(torch.complex64).reshape((n_x, n_y))

plt.figure()
plt.imshow(torch.abs(AHd_image).numpy().T, cmap="gray")
plt.title(
    f"$\\mathbf{{A}}^\\mathrm{{H}} \\mathbf{{d}}$ — Zero-filled Reconstruction ({case})"
)
plt.show()

## Task 5: Iterative Reconstruction with Conjugate Gradient
 
Run CG to solve $\mathbf{Q}\mathbf{x} = \mathbf{b}$ where $\mathbf{Q} = \mathbf{A}^\mathrm{H}\mathbf{A}$ and $\mathbf{b} = \mathbf{A}^\mathrm{H}\mathbf{d}$.
 
**Your tasks:**
 
1. Construct $\mathbf{Q} = \mathbf{A}^\mathrm{H}\mathbf{A}$ and set $\mathbf{b} = \mathbf{A}^\mathrm{H}\mathbf{d}$.
2. Initialize $\mathbf{x}$ as a zero tensor of shape $(2 \cdot n_x \cdot n_y,)$ with `dtype=torch.float32`.
3. Run `conjugate_gradient(Q, b, x, maxiter=30)`.

In [ ]:
# Your solution here


Q = A.H @ A

b = AHd
x = torch.zeros(n_x * n_y, dtype=torch.complex64).view(torch.float32)

x_recon = conjugate_gradient(Q, b, x, maxiter=30)

x_recon_image = x_recon.view(torch.complex64).reshape((n_x, n_y))

## Visualization

In [ ]:
plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(torch.abs(AHd_image).numpy().T, cmap="gray")
plt.title(f"Zero-filled ({case})")
plt.subplot(1, 2, 2)
plt.imshow(torch.abs(x_recon_image).numpy().T, cmap="gray")
plt.title(f"CG Reconstruction ({case})")
plt.tight_layout()
plt.show()

## Task 6: Gradient Operator

The CG reconstruction minimizes $\frac{1}{2}\|\mathbf{A}\mathbf{x} - \mathbf{d}\|_2^2$, which only enforces data consistency. With heavy undersampling ($R = 4$), this problem is ill-posed and the reconstruction shows streaking artifacts. To suppress them, you will add a **Total Variation (TV)** regularization term that promotes images with sparse spatial gradients — i.e. images that are piecewise smooth.
 
TV regularization requires the discrete image gradient operator. The **finite difference gradient** along a spatial dimension $d$ is defined as:
 
$$(\mathbf{G}_d \mathbf{x})_i = x_i - x_{i-1}$$
 
i.e. the difference between each pixel and its predecessor along dimension $d$, with circular boundary conditions. Its adjoint is:
 
$$(\mathbf{G}_d^\mathrm{H} \mathbf{y})_i = y_i - y_{i+1}$$
 
**Your task:** Implement `GradientOperator`. The constructor takes `dim` (the spatial dimension to differentiate along) and `data_shape`. Both `_matvec` and `_rmatvec` operate on tensors of the same shape, so the operator is square. Use `torch.roll` to implement the circular shifts.

In [ ]:
# Your solution here


class GradientOperator(LinearOperator):

    def __init__(
        self,
        dim: int,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        self.dim = dim
        self.device = device
        self.internal_dtype = torch.complex64
        self.dtype = torch.float32
        self.forward_shape = data_shape
        self.adjoint_shape = data_shape
        self.shape = (
            int(2 * torch.prod(torch.tensor(data_shape))),
            int(2 * torch.prod(torch.tensor(data_shape))),
        )

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.forward_shape)
        y = x - torch.roll(x, 1, self.dim)
        return y.view(torch.float32).ravel()

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.adjoint_shape)
        y = x - torch.roll(x, -1, self.dim)
        return y.view(torch.float32).ravel()

## Verify Gradient Operator

Run this cell to visualize the gradient of the CG reconstruction. You should see the edges of the brain highlighted along each axis.

In [ ]:
Gx = GradientOperator(0, (n_x, n_y))
Gy = GradientOperator(1, (n_x, n_y))

gx = Gx @ x_recon
gy = Gy @ x_recon

gx_image = gx.view(torch.complex64).reshape((n_x, n_y))
gy_image = gy.view(torch.complex64).reshape((n_x, n_y))

plt.figure(figsize=(10, 3))

plt.subplot(1, 2, 1)
plt.title(r"$|G_x \mathbf{x}|$")
plt.imshow(
    torch.abs(gx_image).numpy().T,
    cmap="gray",
    vmin=0.0,
    vmax=25,
)
plt.axis("off")
plt.colorbar(label="Schwächungskoeffizient [1/cm]")

plt.subplot(1, 2, 2)
plt.title(r"$|G_y \mathbf{x}|$")
plt.imshow(
    torch.abs(gy_image).numpy().T,
    cmap="gray",
    vmin=0,
    vmax=25,
)
plt.axis("off")
plt.colorbar(label="Schwächungskoeffizient [1/cm]")

## Task 7: Proximal Operator and ADMM

With the gradient operators in place, you can now formulate and solve the **Compressed Sensing** reconstruction problem. The objective is:
 
$$\min_{\mathbf{x}} \;\frac{1}{2} \|\mathbf{A}\mathbf{x} - \mathbf{d}\|_2^2 + \lambda \|\mathbf{G}\mathbf{x}\|_{2,1}$$
 
where $\mathbf{G} = [\mathbf{G}_x;\ \mathbf{G}_y]$ is the joint gradient operator (x and y gradients stacked vertically), and $\|\mathbf{G}\mathbf{x}\|_{2,1}$ is the **isotropic Total Variation** norm — the sum over all pixels of the $\ell_2$ norm of the gradient vector at each pixel. The parameter $\lambda > 0$ controls the strength of the regularization.
 
`ConcatOperator` stacks multiple operators vertically into a single operator. For two operators $\mathbf{G}_x$ and $\mathbf{G}_y$ of the same input and output shape, `G = ConcatOperator((Gx, Gy))` creates $\mathbf{G}$ such that $\mathbf{G}\mathbf{x} = [\mathbf{G}_x \mathbf{x};\ \mathbf{G}_y \mathbf{x}]$.
 
**ADMM** solves the above by introducing the splitting variable $\mathbf{z} = \mathbf{G}\mathbf{x}$ and a scaled dual variable $\mathbf{u}$, then alternating between three update steps:
 
$$\mathbf{x} \leftarrow (\mathbf{Q} + \rho\, \mathbf{G}^\mathrm{H}\mathbf{G})^{-1}(\mathbf{b} + \rho\, \mathbf{G}^\mathrm{H}(\mathbf{z} - \mathbf{u}))$$
 
$$\mathbf{z} \leftarrow \text{prox}_{\lambda/\rho}\,(\mathbf{G}\mathbf{x} + \mathbf{u})$$
 
$$\mathbf{u} \leftarrow \mathbf{u} + \mathbf{G}\mathbf{x} - \mathbf{z}$$
 
**$\mathbf{x}$-update.** This is a linear system of the form $(\mathbf{Q} + \rho\,\mathbf{G}^\mathrm{H}\mathbf{G})\,\mathbf{x} = \mathbf{b} + \rho\,\mathbf{G}^\mathrm{H}(\mathbf{z} - \mathbf{u})$, which you solve with CG — exactly as in Tasks 4 and 5, just with a modified left-hand-side operator and right-hand side. Because ADMM converges over many outer iterations, the $\mathbf{x}$-update does not need to be solved exactly: a small number of CG steps (e.g. `maxiter_cg=5`) is sufficient. This is much cheaper than running CG to convergence at every outer iteration. Note also that the current $\mathbf{x}$ from the previous ADMM iteration is passed as the initial guess — this **warm start** means CG only needs to refine the solution slightly each time rather than starting from scratch.
 
**$\mathbf{z}$-update (joint soft-thresholding).** The proximal operator of the isotropic TV norm acts pixelwise on the stacked gradient $\mathbf{g} = \mathbf{G}\mathbf{x} + \mathbf{u}$ of shape $(2,\ n_x,\ n_y)$. For each pixel $i$, compute the $\ell_2$ norm of its gradient vector:
 
$$r_i = \sqrt{g_{x,i}^2 + g_{y,i}^2}$$
 
then shrink toward zero by $\lambda/\rho$:
 
$$z_i \leftarrow \frac{\max(r_i - \lambda/\rho,\ 0)}{r_i} \cdot g_i$$
 
Use `torch.fmax(r, torch.tensor(lamda))` to clamp `r` before using it in both the numerator and denominator — this avoids division by zero for pixels where the gradient is already below the threshold. Note that `lamda` here is already the scaled threshold $\lambda/\rho$, since the caller passes `lamda/rho` when invoking `joint_soft_thresholding`.
 
**Your tasks:**
 
1. Implement `joint_soft_thresholding(g, lamda, shape)`. The input `g` is a flat real tensor. Reshape it to `shape = (2, n_x, n_y)`, compute $r$ with `torch.sqrt(torch.sum(g**2, axis=0))`, apply the shrinkage formula, and return the result raveled.
2. Implement `admm(Q, b, G, x, lamda, rho, shape, maxiter_admm, maxiter_cg)`. Initialize $\mathbf{z}$ and $\mathbf{u}$ as zero tensors of size `G.shape[0]` with `dtype=x.dtype`. Then iterate the three update steps, using `conjugate_gradient` with `disable_progressbar=True` and passing the current `x` as the initial guess (warm start).

In [ ]:
# Your solution here


def joint_soft_thresholding(
    g: np.ndarray,
    lamda: float,
    shape: Tuple[int, int],
) -> torch.tensor:

    g = torch.reshape(g, shape)

    r = torch.sqrt(torch.sum(g**2, axis=0))

    r = torch.fmax(r, torch.tensor((lamda,)))

    factor = (r - lamda) / r

    g = factor * g

    return g.ravel()


def admm(
    Q: LinearOperator,
    b: torch.tensor,
    G: LinearOperator,
    x: torch.tensor,
    lamda: float,
    rho: float,
    shape: Tuple[int, int],
    maxiter_admm: int = 30,
    maxiter_cg: int = 5,
) -> torch.tensor:

    z = torch.zeros(G.shape[0], dtype=x.dtype)
    u = torch.zeros(G.shape[0], dtype=x.dtype)

    for iteration in tqdm(range(maxiter_admm)):

        x = conjugate_gradient(
            Q + rho * G.H @ G,
            b=b + rho * G.H @ (z - u),
            x=x,
            maxiter=maxiter_cg,
            disable_progressbar=True,
        )

        z = joint_soft_thresholding(
            G @ x + u,
            lamda / rho,
            shape,
        )

        u = u + G @ x - z

    return x

## Task 8: Compressed Sensing Reconstruction with ADMM
 
You now have all the pieces. Construct the joint gradient operator $\mathbf{G} = [\mathbf{G}_x;\ \mathbf{G}_y]$ using `ConcatOperator` and run ADMM to solve the TV-regularized reconstruction problem.
 
The `shape` argument to `admm` describes the shape of the stacked gradient output: `(2, n_x, n_y)` since $\mathbf{G}$ stacks two gradient images. The inner `shape` argument to `joint_soft_thresholding` receives `(2, *shape)` — think about why.
 
**Your tasks:**
 
1. Instantiate `Gx = GradientOperator(0, (n_x, n_y))` and `Gy = GradientOperator(1, (n_x, n_y))`.
2. Construct `G = ConcatOperator((Gx, Gy))`.
3. Set `Q = A.H @ A`, `b = AHd`, and initialize `x` as zeros.
4. Call `admm(Q, b, G, x, lamda=5e-2, rho=1, shape=(2, n_x, n_y), maxiter_admm=100, maxiter_cg=5)`.
5. Reshape the result and display it in the comparison plot below.

In [ ]:
# Your solution here

Gx = GradientOperator(0, (n_x, n_y))
Gy = GradientOperator(1, (n_x, n_y))
G = ConcatOperator((Gx, Gy))
shape = (2, n_x, n_y)

Q = A.H @ A

b = AHd
x = torch.zeros(
    n_x * n_y,
    dtype=torch.complex64,
).view(torch.float32)

# Reconstruction with ADMM
x_admm = admm(
    Q,
    b,
    G,
    x,
    lamda=5e-2,
    rho=1,
    shape=(2, *shape),
    maxiter_admm=100,
    maxiter_cg=5,
)

x_admm_image = x_admm.view(torch.complex64).reshape((n_x, n_y))

## Comparison

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(1, 3, 1)
plt.imshow(torch.abs(AHd_image).numpy().T, cmap="gray")
plt.title(f"Zero-filled ({case})")
plt.subplot(1, 3, 2)
plt.imshow(torch.abs(x_recon_image).numpy().T, cmap="gray")
plt.title(f"CG Reconstruction ({case})")
plt.subplot(1, 3, 3)
plt.imshow(torch.abs(x_admm_image).numpy().T, cmap="gray")
plt.title(f"CS Reconstruction ({case})")
plt.tight_layout()
plt.show()